In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams


def read_rms_from_log(log_file):
    """
    从 run.log 中提取迭代号和 RMS_ZH
    适配格式：
    Iteration log
    iter    RMS_ZH      m_total      nar
    --------------------------------------
       1    0.123456    ...
       2    0.101234    ...
    """
    iters = []
    rms = []

    data_line_pattern = re.compile(
        r"^\s*(\d+)\s+([-+]?\d*\.?\d+(?:[Ee][-+]?\d+)?)\s+(\d+)\s+(\d+)\s*$"
    )

    with open(log_file, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    in_table = False
    for line in lines:
        if "Iteration log" in line:
            in_table = True
            continue
        if not in_table:
            continue

        m = data_line_pattern.match(line.strip())
        if m:
            iters.append(int(m.group(1)))
            rms.append(float(m.group(2)))

    if len(iters) == 0:
        raise ValueError(f"没有在 {log_file} 中识别到 RMS 迭代表格，请检查 log 格式。")

    return np.array(iters), np.array(rms)


def plot_rms_curve(iters, rms, out_png=None, out_pdf=None,
                   title="Residual reduction during iterations"):
    """
    论文风格绘制残差曲线
    """
    rcParams["font.family"] = "Times New Roman"
    rcParams["font.size"] = 12
    rcParams["axes.labelsize"] = 14
    rcParams["axes.titlesize"] = 14
    rcParams["xtick.labelsize"] = 12
    rcParams["ytick.labelsize"] = 12
    rcParams["legend.fontsize"] = 11

    fig, ax = plt.subplots(figsize=(6.2, 4.5))

    ax.plot(iters, rms, "-o", lw=1.8, ms=5.5, mec="black")

    # 标出起点和终点
    ax.scatter(iters[0], rms[0], s=45, zorder=3)
    ax.scatter(iters[-1], rms[-1], s=45, zorder=3)

    ax.set_xlabel("Iteration")
    ax.set_ylabel("RMS misfit")
    ax.set_title(title)

    ax.grid(True, alpha=0.25, linestyle="--", linewidth=0.6)

    # 边框稍微粗一点
    for spine in ax.spines.values():
        spine.set_linewidth(1.0)

    # x 轴整数刻度
    if len(iters) <= 20:
        ax.set_xticks(iters)

    # 在图中写下降百分比
    reduction = (rms[0] - rms[-1]) / rms[0] * 100.0 if rms[0] != 0 else 0.0
    text = (
        f"Initial RMS = {rms[0]:.4f}\n"
        f"Final RMS = {rms[-1]:.4f}\n"
        f"Reduction = {reduction:.1f}%"
    )
    ax.text(
        0.98, 0.97, text,
        transform=ax.transAxes,
        ha="right", va="top",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.85, edgecolor="0.5")
    )

    plt.tight_layout()

    if out_png:
        plt.savefig(out_png, dpi=300, bbox_inches="tight")
        print(f"已保存: {out_png}")
    if out_pdf:
        plt.savefig(out_pdf, bbox_inches="tight")
        print(f"已保存: {out_pdf}")

    plt.show()


if __name__ == "__main__":
    # ===== 改这里 =====
    log_file = "real_run.log"
    out_png = "rms_curve.png"
    out_pdf = "rms_curve.pdf"
    title = "Residual reduction during ZH inversion"

    iters, rms = read_rms_from_log(log_file)
    print("iters =", iters)
    print("rms   =", rms)

    plot_rms_curve(
        iters, rms,
        out_png=out_png,
        out_pdf=out_pdf,
        title=title
    )